# AI-Assisted Business Analysis: Experimental Pipeline

This notebook contains the reproducible generation and stability-analysis pipeline used in the dissertation:

**AI-Assisted Business Analysis: An Empirical Evaluation of LLM-Generated Business Analysis Artifacts Across Prompting Approaches**

The experiment compares four prompting approaches (Zero-shot, Role-based, Structured, and Few-shot) across three Business Analysis artifact types (User Stories, Acceptance Criteria, and Business Rules) using 10 FinTech scenarios and three repeated generations per condition.

**Formal design:** 10 scenarios × 3 artifacts × 4 prompting approaches × 3 generations = **360 outputs**.

The notebook intentionally excludes the historical pilot workflow and participant-level human-evaluation data. It is designed to run against the public research workbook included with the repository.


## 1. Install dependencies

The versions can also be pinned in the repository `requirements.txt`.


In [ ]:
!pip install -q --upgrade openai openpyxl


## 2. Imports and configuration

For local execution, set the `OPENAI_API_KEY` environment variable.

For Google Colab, the notebook can read `OPENAI_API_KEY` from **Colab Secrets**. The API key is never stored in the notebook or workbook.


In [ ]:
import itertools
import json
import os
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from openai import OpenAI
from openpyxl import load_workbook
from scipy.stats import friedmanchisquare, rankdata, wilcoxon

MODEL_ID = "gpt-5.6-terra"
REASONING_EFFORT = "low"
EMBEDDING_MODEL = "text-embedding-3-large"

WORKBOOK_PATH = Path("data/Thesis_Experiment_Public.xlsx")
EMBEDDING_CACHE_PATH = Path("data/formal_embedding_cache.json")

# Safety switches: keep False unless you intentionally want to incur API usage.
RUN_GENERATION = False
RUN_EMBEDDINGS = False

print("Model:", MODEL_ID)
print("Reasoning effort:", REASONING_EFFORT)
print("Embedding model:", EMBEDDING_MODEL)
print("Workbook:", WORKBOOK_PATH)


## 3. API authentication

The code first checks the standard environment variable. If it is not available and the notebook is running in Colab, it attempts to read the key from Colab Secrets.


In [ ]:
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    try:
        from google.colab import userdata
        api_key = userdata.get("OPENAI_API_KEY")
    except Exception:
        api_key = None

if not api_key:
    raise ValueError(
        "OPENAI_API_KEY was not found. "
        "Set it as an environment variable or add it to Colab Secrets."
    )

client = OpenAI(api_key=api_key)
print("OpenAI client initialised. API key was not printed or stored.")


## 4. Validate and load the research workbook

The public workbook must contain the scenario cards, prompt library, few-shot examples, experiment matrix, API-results schema, and similarity schema used by this pipeline.


In [ ]:
REQUIRED_SHEETS = [
    "02_Scenario_Cards",
    "05_Prompt_Library",
    "06_FewShot_Examples",
    "07_Experiment_Matrix",
    "08_API_Results",
    "11_Similarity",
]

if not WORKBOOK_PATH.exists():
    raise FileNotFoundError(
        f"Workbook not found: {WORKBOOK_PATH}. "
        "Clone the repository or place the public workbook in the data/ folder."
    )

excel_file = pd.ExcelFile(WORKBOOK_PATH)
missing_sheets = [s for s in REQUIRED_SHEETS if s not in excel_file.sheet_names]

if missing_sheets:
    raise ValueError(f"Missing required workbook sheets: {missing_sheets}")

scenario_df = pd.read_excel(WORKBOOK_PATH, sheet_name="02_Scenario_Cards")
prompt_df = pd.read_excel(WORKBOOK_PATH, sheet_name="05_Prompt_Library")
fewshot_df = pd.read_excel(WORKBOOK_PATH, sheet_name="06_FewShot_Examples")
experiment_df = pd.read_excel(WORKBOOK_PATH, sheet_name="07_Experiment_Matrix")

print("Workbook validation passed.")
print("Experiment rows:", len(experiment_df))


## 5. Prompt construction

The scenario content is held constant across comparable experimental conditions. Prompt templates vary only according to the defined prompting approach. Two additional scenarios are used only as Few-shot demonstrations.


In [ ]:
def get_scenario_text(scenario_id: str) -> str:
    matches = scenario_df.loc[
        scenario_df["Scenario ID"] == scenario_id,
        "Prompt Scenario Text",
    ]
    if len(matches) != 1:
        raise ValueError(
            f"Exactly one scenario card is required for {scenario_id}; found {len(matches)}."
        )
    value = matches.iloc[0]
    if pd.isna(value) or not str(value).strip():
        raise ValueError(f"Prompt Scenario Text is empty for {scenario_id}.")
    return str(value).strip()


def build_fewshot_examples(artifact: str) -> str:
    examples = fewshot_df.loc[fewshot_df["Artifact"] == artifact].copy()
    if len(examples) != 2:
        raise ValueError(
            f"Exactly two Few-shot examples are required for {artifact}; found {len(examples)}."
        )

    formatted = []
    for number, (_, row) in enumerate(examples.iterrows(), start=1):
        scenario_text = get_scenario_text(str(row["Scenario ID"]))
        example_output = str(row["Approved Example Output"]).strip()
        formatted.append(
            f"EXAMPLE {number}\n\n"
            f"Scenario:\n{scenario_text}\n\n"
            f"{artifact}:\n{example_output}"
        )

    return "\n\n------------------------------\n\n".join(formatted)


def build_final_prompt(scenario_id: str, prompt_id: str) -> dict[str, Any]:
    matches = prompt_df.loc[prompt_df["Prompt ID"] == prompt_id]
    if len(matches) != 1:
        raise ValueError(
            f"Exactly one prompt template is required for {prompt_id}; found {len(matches)}."
        )

    row = matches.iloc[0]
    artifact = str(row["Artifact"]).strip()
    technique = str(row["Technique"]).strip()
    template = str(row["Prompt Template"])

    final_prompt = template.replace("{SCENARIO_TEXT}", get_scenario_text(scenario_id))

    if technique == "Few-shot":
        placeholder_map = {
            "User Story": "{FEW_SHOT_EXAMPLES_USER_STORY}",
            "Acceptance Criteria": "{FEW_SHOT_EXAMPLES_ACCEPTANCE_CRITERIA}",
            "Business Rules": "{FEW_SHOT_EXAMPLES_BUSINESS_RULES}",
        }
        final_prompt = final_prompt.replace(
            placeholder_map[artifact],
            build_fewshot_examples(artifact),
        )

    if "{" in final_prompt or "}" in final_prompt:
        raise ValueError(f"Unresolved placeholder detected in prompt {prompt_id}.")

    return {
        "scenario_id": scenario_id,
        "prompt_id": prompt_id,
        "artifact": artifact,
        "technique": technique,
        "final_prompt": final_prompt.strip(),
    }


print("Prompt builder ready.")


## 6. Workbook checkpoint helpers

Each completed API response is written immediately to the workbook. Existing completed runs are skipped, allowing interrupted experiments to resume safely.


In [ ]:
def safe_getattr(obj: Any, attribute: str, default: Any = None) -> Any:
    if obj is None:
        return default
    return getattr(obj, attribute, default)


def get_header_map(worksheet) -> dict[str, int]:
    return {
        str(cell.value).strip(): cell.column
        for cell in worksheet[1]
        if cell.value is not None
    }


def find_row_by_value(worksheet, header_name: str, target_value: str) -> int:
    headers = get_header_map(worksheet)
    if header_name not in headers:
        raise KeyError(f"'{header_name}' not found in sheet '{worksheet.title}'.")

    column_number = headers[header_name]
    for row_number in range(2, worksheet.max_row + 1):
        value = worksheet.cell(row=row_number, column=column_number).value
        if str(value).strip() == str(target_value).strip():
            return row_number

    raise ValueError(
        f"{header_name}={target_value} was not found in sheet '{worksheet.title}'."
    )


def atomic_save_workbook(workbook, workbook_path: Path) -> None:
    temporary_path = workbook_path.with_name(
        workbook_path.stem + "_temporary_save.xlsx"
    )
    workbook.save(temporary_path)
    os.replace(temporary_path, workbook_path)


def get_experiment_run_status(output_run_id: str) -> str:
    workbook = load_workbook(WORKBOOK_PATH, data_only=False)
    worksheet = workbook["07_Experiment_Matrix"]
    headers = get_header_map(worksheet)
    row_number = find_row_by_value(
        worksheet, "Output Run ID", output_run_id
    )
    status = worksheet.cell(
        row=row_number, column=headers["Run Status"]
    ).value
    workbook.close()
    return "" if status is None else str(status).strip()


def write_result_to_workbook(result: dict[str, Any]) -> None:
    workbook = load_workbook(WORKBOOK_PATH, data_only=False)
    experiment_ws = workbook["07_Experiment_Matrix"]
    api_ws = workbook["08_API_Results"]

    experiment_headers = get_header_map(experiment_ws)
    api_headers = get_header_map(api_ws)

    output_run_id = result["output_run_id"]
    experiment_row = find_row_by_value(
        experiment_ws, "Output Run ID", output_run_id
    )
    api_row = find_row_by_value(
        api_ws, "Output Run ID", output_run_id
    )

    api_values = {
        "Output Run ID": result.get("output_run_id"),
        "Condition ID": result.get("condition_id"),
        "Scenario ID": result.get("scenario_id"),
        "Artifact": result.get("artifact"),
        "Technique": result.get("technique"),
        "Prompt ID": result.get("prompt_id"),
        "Generation": result.get("generation"),
        "Model ID": result.get("model", MODEL_ID),
        "Reasoning Mode": "standard",
        "Reasoning Effort": REASONING_EFFORT,
        "Request ID": result.get("request_id"),
        "Start Time UTC": result.get("start_time_utc"),
        "End Time UTC": result.get("end_time_utc"),
        "Latency (s)": result.get("latency_seconds"),
        "Input Tokens": result.get("input_tokens"),
        "Cached Input Tokens": result.get("cached_input_tokens"),
        "Output Tokens": result.get("output_tokens"),
        "Reasoning Tokens": result.get("reasoning_tokens"),
        # "Total Tokens" and "Estimated Cost (USD)" remain workbook formulas.
        "Raw Output": result.get("raw_output"),
        "API Status": result.get("api_status"),
        "Error / Notes": result.get("error"),
    }

    for header_name, value in api_values.items():
        if header_name not in api_headers:
            raise KeyError(
                f"'{header_name}' not found in 08_API_Results."
            )
        api_ws.cell(
            row=api_row,
            column=api_headers[header_name],
        ).value = value

    experiment_ws.cell(
        row=experiment_row,
        column=experiment_headers["Run Status"],
    ).value = (
        "Completed" if result.get("api_status") == "Completed" else "Failed"
    )

    atomic_save_workbook(workbook, WORKBOOK_PATH)
    workbook.close()


print("Workbook checkpoint helpers ready.")


## 7. LLM generation

Each experimental condition is executed as an independent API call with `store=False`. No conversation history is passed between generations.


In [ ]:
def create_api_result(
    experiment_row: pd.Series,
    max_attempts: int = 3,
) -> dict[str, Any]:
    output_run_id = str(experiment_row["Output Run ID"])
    condition_id = str(experiment_row["Condition ID"])
    scenario_id = str(experiment_row["Scenario ID"])
    prompt_id = str(experiment_row["Prompt ID"])
    generation = int(experiment_row["Generation"])

    prompt_data = build_final_prompt(
        scenario_id=scenario_id,
        prompt_id=prompt_id,
    )

    last_error = None

    for attempt in range(1, max_attempts + 1):
        start_time_utc = datetime.now(timezone.utc)
        start_counter = time.perf_counter()

        try:
            response = client.responses.create(
                model=MODEL_ID,
                reasoning={"effort": REASONING_EFFORT},
                input=prompt_data["final_prompt"],
                store=False,
            )

            latency_seconds = time.perf_counter() - start_counter
            end_time_utc = datetime.now(timezone.utc)
            usage = response.usage

            input_tokens = safe_getattr(usage, "input_tokens", 0)
            output_tokens = safe_getattr(usage, "output_tokens", 0)
            input_details = safe_getattr(usage, "input_tokens_details")
            output_details = safe_getattr(usage, "output_tokens_details")
            cached_input_tokens = safe_getattr(input_details, "cached_tokens", 0)
            reasoning_tokens = safe_getattr(output_details, "reasoning_tokens", 0)

            raw_output = (response.output_text or "").strip()
            if not raw_output:
                raise ValueError("The API returned an empty output.")

            return {
                "output_run_id": output_run_id,
                "condition_id": condition_id,
                "scenario_id": scenario_id,
                "artifact": prompt_data["artifact"],
                "technique": prompt_data["technique"],
                "prompt_id": prompt_id,
                "generation": generation,
                "model": response.model,
                "request_id": response.id,
                "start_time_utc": start_time_utc.isoformat(),
                "end_time_utc": end_time_utc.isoformat(),
                "latency_seconds": latency_seconds,
                "input_tokens": input_tokens,
                "cached_input_tokens": cached_input_tokens,
                "output_tokens": output_tokens,
                "reasoning_tokens": reasoning_tokens,
                "raw_output": raw_output,
                "api_status": "Completed",
                "error": None,
            }

        except Exception as exc:
            last_error = f"{type(exc).__name__}: {exc}"
            print(
                f"{output_run_id}: attempt {attempt}/{max_attempts} failed: {last_error}"
            )
            if attempt < max_attempts:
                time.sleep(2 ** attempt)

    return {
        "output_run_id": output_run_id,
        "condition_id": condition_id,
        "scenario_id": scenario_id,
        "artifact": prompt_data["artifact"],
        "technique": prompt_data["technique"],
        "prompt_id": prompt_id,
        "generation": generation,
        "model": MODEL_ID,
        "start_time_utc": None,
        "end_time_utc": datetime.now(timezone.utc).isoformat(),
        "latency_seconds": None,
        "input_tokens": None,
        "cached_input_tokens": None,
        "output_tokens": None,
        "reasoning_tokens": None,
        "raw_output": None,
        "api_status": "Failed",
        "error": last_error,
    }


def run_formal_experiment() -> None:
    latest = pd.read_excel(
        WORKBOOK_PATH,
        sheet_name="07_Experiment_Matrix",
    )

    formal_runs = latest.loc[
        latest["Dataset Role"] == "Evaluation"
    ].sort_values("Output Run ID").reset_index(drop=True)

    if len(formal_runs) != 360:
        raise ValueError(
            f"Expected 360 formal runs; found {len(formal_runs)}."
        )

    skipped = completed_now = failed_now = 0

    for index, row in formal_runs.iterrows():
        output_run_id = str(row["Output Run ID"])

        if get_experiment_run_status(output_run_id) == "Completed":
            skipped += 1
            print(f"[{index + 1:03d}/360] {output_run_id} - completed, skipped.")
            continue

        print(
            f"[{index + 1:03d}/360] {output_run_id} | "
            f"{row['Scenario ID']} | {row['Artifact']} | "
            f"{row['Technique']} | Generation {row['Generation']}"
        )

        result = create_api_result(row)
        write_result_to_workbook(result)

        if result["api_status"] == "Completed":
            completed_now += 1
        else:
            failed_now += 1

        time.sleep(0.5)

    print("Previously completed:", skipped)
    print("Completed now:", completed_now)
    print("Failed now:", failed_now)


if RUN_GENERATION:
    run_formal_experiment()
else:
    print(
        "Generation is disabled by default. "
        "Set RUN_GENERATION = True only if you intend to make API calls."
    )


## 8. Validate formal generation

The original experiment contained 360 completed outputs with no empty completed responses.


In [ ]:
api_results_df = pd.read_excel(
    WORKBOOK_PATH,
    sheet_name="08_API_Results",
)

completed_outputs = api_results_df.loc[
    api_results_df["API Status"] == "Completed"
].copy()

empty_outputs = completed_outputs.loc[
    completed_outputs["Raw Output"].fillna("").astype(str).str.strip().eq("")
]

print("Completed outputs:", len(completed_outputs))
print("Empty completed outputs:", len(empty_outputs))

if len(completed_outputs) == 360 and len(empty_outputs) == 0:
    print("FORMAL GENERATION VALIDATION PASSED.")
else:
    print(
        "The workbook does not currently contain the complete original "
        "360-output generation set. This is expected if the public workbook "
        "was released without raw generated outputs."
    )


## 9. Semantic stability analysis

Stability is measured using `text-embedding-3-large`. For each condition, cosine similarity is calculated for Run 1–Run 2, Run 1–Run 3, and Run 2–Run 3. Their average is the **Mean Pairwise Similarity**.

Semantic similarity is treated as a **repeatability/stability metric, not a quality metric**.

The public workbook includes the original pairwise similarity results in `11_Similarity`. If the generation and embedding stages are rerun, the code below recalculates the 120 condition-level similarity records and overwrites that sheet with the reproduced values.


In [ ]:
def load_embedding_cache() -> dict:
    if not EMBEDDING_CACHE_PATH.exists():
        return {}
    with open(EMBEDDING_CACHE_PATH, "r", encoding="utf-8") as file:
        return json.load(file)


def save_embedding_cache(cache: dict) -> None:
    EMBEDDING_CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = EMBEDDING_CACHE_PATH.with_suffix(".temporary.json")
    with open(temporary_path, "w", encoding="utf-8") as file:
        json.dump(cache, file)
    os.replace(temporary_path, EMBEDDING_CACHE_PATH)


def generate_missing_embeddings(
    outputs_df: pd.DataFrame,
    batch_size: int = 100,
) -> dict:
    cache = load_embedding_cache()
    missing_rows = outputs_df.loc[
        ~outputs_df["Output Run ID"].astype(str).isin(cache.keys())
    ].copy()

    print("New embeddings required:", len(missing_rows))

    records = missing_rows[
        ["Output Run ID", "Raw Output"]
    ].to_dict("records")

    for start_index in range(0, len(records), batch_size):
        batch = records[start_index:start_index + batch_size]
        run_ids = [str(record["Output Run ID"]) for record in batch]
        texts = [str(record["Raw Output"]).strip() for record in batch]

        response = client.embeddings.create(
            model=EMBEDDING_MODEL,
            input=texts,
            encoding_format="float",
        )

        if len(response.data) != len(batch):
            raise RuntimeError("Embedding response count did not match batch size.")

        for run_id, item in zip(run_ids, response.data):
            cache[run_id] = item.embedding

        save_embedding_cache(cache)

    return cache


def cosine_similarity(vector_a: list[float], vector_b: list[float]) -> float:
    a = np.asarray(vector_a, dtype=np.float64)
    b = np.asarray(vector_b, dtype=np.float64)
    denominator = np.linalg.norm(a) * np.linalg.norm(b)

    if denominator == 0:
        raise ValueError("A zero-length embedding vector was found.")

    return float(np.dot(a, b) / denominator)


def write_similarity_results_to_workbook(similarity_df: pd.DataFrame) -> None:
    expected_columns = [
        "Condition ID",
        "Scenario ID",
        "Artifact",
        "Technique",
        "Run 1 ID",
        "Run 2 ID",
        "Run 3 ID",
        "Similarity R1–R2",
        "Similarity R1–R3",
        "Similarity R2–R3",
        "Mean Pairwise Similarity",
        "Embedding Model",
        "Method Notes",
    ]

    if similarity_df.columns.tolist() != expected_columns:
        raise ValueError(
            "Similarity output columns do not match the public workbook schema."
        )

    workbook = load_workbook(WORKBOOK_PATH)
    worksheet = workbook["11_Similarity"]

    # Preserve the header row and replace only the data rows.
    if worksheet.max_row > 1:
        worksheet.delete_rows(2, worksheet.max_row - 1)

    for row in similarity_df.itertuples(index=False, name=None):
        worksheet.append(list(row))

    atomic_save_workbook(workbook, WORKBOOK_PATH)
    workbook.close()
    print("11_Similarity updated with reproduced results.")


formal_outputs = api_results_df.loc[
    api_results_df["API Status"].eq("Completed")
].copy()

formal_outputs["Raw Output"] = (
    formal_outputs["Raw Output"].fillna("").astype(str).str.strip()
)

if RUN_EMBEDDINGS:
    if len(formal_outputs) != 360:
        raise ValueError(
            "Embedding reproduction requires the complete 360-output dataset."
        )

    embedding_cache = generate_missing_embeddings(formal_outputs)

    similarity_records = []

    for condition_id, group in formal_outputs.groupby("Condition ID"):
        group = group.sort_values("Generation").reset_index(drop=True)

        if group["Generation"].tolist() != [1, 2, 3]:
            raise ValueError(
                f"Condition {condition_id} does not contain generations 1, 2, and 3."
            )

        run_1 = str(group.loc[0, "Output Run ID"])
        run_2 = str(group.loc[1, "Output Run ID"])
        run_3 = str(group.loc[2, "Output Run ID"])

        sim_12 = cosine_similarity(embedding_cache[run_1], embedding_cache[run_2])
        sim_13 = cosine_similarity(embedding_cache[run_1], embedding_cache[run_3])
        sim_23 = cosine_similarity(embedding_cache[run_2], embedding_cache[run_3])

        similarity_records.append({
            "Condition ID": condition_id,
            "Scenario ID": group.loc[0, "Scenario ID"],
            "Artifact": group.loc[0, "Artifact"],
            "Technique": group.loc[0, "Technique"],
            "Run 1 ID": run_1,
            "Run 2 ID": run_2,
            "Run 3 ID": run_3,
            "Similarity R1–R2": sim_12,
            "Similarity R1–R3": sim_13,
            "Similarity R2–R3": sim_23,
            "Mean Pairwise Similarity": float(np.mean([sim_12, sim_13, sim_23])),
            "Embedding Model": EMBEDDING_MODEL,
            "Method Notes": (
                "Cosine similarity between OpenAI text-embedding-3-large vectors; "
                "stability metric, not quality metric."
            ),
        })

    similarity_df = (
        pd.DataFrame(similarity_records)
        .sort_values("Condition ID")
        .reset_index(drop=True)
    )

    if len(similarity_df) != 120:
        raise ValueError(
            f"Expected 120 similarity records; found {len(similarity_df)}."
        )

    write_similarity_results_to_workbook(similarity_df)
    display(similarity_df.head())
else:
    print(
        "Embedding calls are disabled by default. "
        "Set RUN_EMBEDDINGS = True only when the complete generated-output "
        "dataset is available and you intend to make embedding API calls."
    )


## 10. Statistical analysis

The public workbook contains condition-level aggregated human-evaluation results in `13_Evaluation_Summary`. These data are sufficient to reproduce the dissertation's main inferential comparisons without making any API calls.

The analysis uses:

- **Friedman repeated-measures tests** for omnibus comparisons.
- **Kendall's W** as the Friedman effect size.
- **Paired Wilcoxon signed-rank tests** for pairwise post-hoc comparisons.
- **Holm adjustment** to control for multiple pairwise comparisons.
- **Rank-biserial correlation** as the paired Wilcoxon effect size.

The tests use **Mean Human Quality**, the condition-level mean of the five human-evaluation criteria after averaging across the three expert evaluators.

Inter-rater reliability is not recalculated in this public notebook because ICC requires evaluator-level ratings. Those records are intentionally excluded from the public release. The resulting ICC statistics are reported in `19_Statistical_Analysis`.

> Interpretation note: Pairwise comparisons are reproduced for completeness, but isolated pairwise significance should not be emphasised when the corresponding Friedman omnibus test is not significant.


In [ ]:
# Load aggregated condition-level human evaluation data.
evaluation_df = pd.read_excel(
    WORKBOOK_PATH,
    sheet_name="13_Evaluation_Summary",
)

required_stat_columns = {
    "Scenario ID",
    "Artifact",
    "Technique",
    "Mean Human Quality",
}
missing_stat_columns = required_stat_columns.difference(evaluation_df.columns)

if missing_stat_columns:
    raise ValueError(
        f"Missing columns required for statistical analysis: "
        f"{sorted(missing_stat_columns)}"
    )

if len(evaluation_df) != 120:
    raise ValueError(
        f"Expected 120 condition-level evaluation rows; found {len(evaluation_df)}."
    )

TECHNIQUE_ORDER = ["Zero-shot", "Role-based", "Structured", "Few-shot"]
ARTIFACT_ORDER = ["User Story", "Acceptance Criteria", "Business Rules"]


def kendalls_w_from_friedman(chi_square, n_blocks, n_levels):
    return float(chi_square / (n_blocks * (n_levels - 1)))


def holm_adjust(p_values):
    """Holm step-down multiple-comparison adjustment."""
    p_values = np.asarray(p_values, dtype=float)
    m = len(p_values)
    order = np.argsort(p_values)

    adjusted = np.empty(m, dtype=float)
    running_max = 0.0

    for rank, original_index in enumerate(order):
        candidate = min(1.0, (m - rank) * p_values[original_index])
        running_max = max(running_max, candidate)
        adjusted[original_index] = running_max

    return adjusted.tolist()


def paired_rank_biserial(x, y):
    """Paired rank-biserial correlation from signed ranks."""
    differences = np.asarray(x, dtype=float) - np.asarray(y, dtype=float)
    differences = differences[differences != 0]

    if len(differences) == 0:
        return 0.0

    ranks = rankdata(np.abs(differences), method="average")
    positive_rank_sum = float(ranks[differences > 0].sum())
    negative_rank_sum = float(ranks[differences < 0].sum())

    return (
        (positive_rank_sum - negative_rank_sum)
        / (positive_rank_sum + negative_rank_sum)
    )


def run_friedman(matrix, level_order, scope, factor):
    ordered = matrix[level_order].dropna()
    arrays = [
        ordered[level].to_numpy(dtype=float)
        for level in level_order
    ]

    result = friedmanchisquare(*arrays)
    n_blocks = len(ordered)
    n_levels = len(level_order)

    return {
        "Scope": scope,
        "Factor": factor,
        "Blocks (n)": n_blocks,
        "Levels": n_levels,
        "Friedman χ²": float(result.statistic),
        "p-value": float(result.pvalue),
        "Kendall's W": kendalls_w_from_friedman(
            float(result.statistic),
            n_blocks,
            n_levels,
        ),
        "Significant (α=.05)": bool(result.pvalue < 0.05),
    }


def run_pairwise_wilcoxon(matrix, level_order, scope, factor):
    ordered = matrix[level_order].dropna()

    records = []
    raw_p_values = []

    for left, right in itertools.combinations(level_order, 2):
        x = ordered[left].to_numpy(dtype=float)
        y = ordered[right].to_numpy(dtype=float)

        result = wilcoxon(
            x,
            y,
            alternative="two-sided",
            zero_method="wilcox",
            correction=False,
            method="auto",
        )

        raw_p = float(result.pvalue)
        raw_p_values.append(raw_p)

        records.append({
            "Scope": scope,
            "Factor": factor,
            "Level A": left,
            "Level B": right,
            "Pairs (n)": len(ordered),
            "Wilcoxon W": float(result.statistic),
            "Raw p": raw_p,
            "Rank-biserial r": paired_rank_biserial(x, y),
        })

    adjusted_p_values = holm_adjust(raw_p_values)

    for record, adjusted_p in zip(records, adjusted_p_values):
        record["Holm-adjusted p"] = float(adjusted_p)
        record["Significant (α=.05)"] = bool(adjusted_p < 0.05)

    return pd.DataFrame(records)


# 1) Overall prompting-technique effect:
#    30 scenario × artifact blocks, 4 prompting approaches.
prompt_overall_matrix = evaluation_df.pivot(
    index=["Scenario ID", "Artifact"],
    columns="Technique",
    values="Mean Human Quality",
)

omnibus_records = [
    run_friedman(
        prompt_overall_matrix,
        TECHNIQUE_ORDER,
        scope="Overall",
        factor="Prompting Technique",
    )
]

posthoc_tables = [
    run_pairwise_wilcoxon(
        prompt_overall_matrix,
        TECHNIQUE_ORDER,
        scope="Overall",
        factor="Prompting Technique",
    )
]


# 2) Prompting-technique effect within each artifact type:
#    10 scenario blocks, 4 prompting approaches.
for artifact in ARTIFACT_ORDER:
    artifact_subset = evaluation_df.loc[
        evaluation_df["Artifact"].eq(artifact)
    ]

    artifact_prompt_matrix = artifact_subset.pivot(
        index="Scenario ID",
        columns="Technique",
        values="Mean Human Quality",
    )

    omnibus_records.append(
        run_friedman(
            artifact_prompt_matrix,
            TECHNIQUE_ORDER,
            scope=artifact,
            factor="Prompting Technique",
        )
    )

    posthoc_tables.append(
        run_pairwise_wilcoxon(
            artifact_prompt_matrix,
            TECHNIQUE_ORDER,
            scope=artifact,
            factor="Prompting Technique",
        )
    )


# 3) Artifact-type effect:
#    40 scenario × technique blocks, 3 artifact types.
artifact_matrix = evaluation_df.pivot(
    index=["Scenario ID", "Technique"],
    columns="Artifact",
    values="Mean Human Quality",
)

omnibus_records.append(
    run_friedman(
        artifact_matrix,
        ARTIFACT_ORDER,
        scope="Overall",
        factor="Artifact Type",
    )
)

posthoc_tables.append(
    run_pairwise_wilcoxon(
        artifact_matrix,
        ARTIFACT_ORDER,
        scope="Overall",
        factor="Artifact Type",
    )
)


omnibus_df = pd.DataFrame(omnibus_records)
posthoc_df = pd.concat(posthoc_tables, ignore_index=True)

print("Omnibus repeated-measures tests")
display(omnibus_df)

print("\nPost-hoc paired Wilcoxon tests with Holm adjustment")
display(posthoc_df)


# Reproducibility checks against the values reported in the dissertation workbook.
expected_checks = {
    ("Overall", "Prompting Technique"): (
        20.501730103806228,
        0.00013358433495427052,
    ),
    ("User Story", "Prompting Technique"): (
        18.359999999999985,
        0.00037069959318386455,
    ),
    ("Acceptance Criteria", "Prompting Technique"): (
        5.680851063829791,
        0.12821308186070343,
    ),
    ("Business Rules", "Prompting Technique"): (
        3.1263157894736833,
        0.3725568108052839,
    ),
    ("Overall", "Artifact Type"): (
        62.33766233766234,
        2.907693435648793e-14,
    ),
}

for (scope, factor), (expected_chi2, expected_p) in expected_checks.items():
    row = omnibus_df.loc[
        omnibus_df["Scope"].eq(scope)
        & omnibus_df["Factor"].eq(factor)
    ].iloc[0]

    if not np.isclose(
        row["Friedman χ²"],
        expected_chi2,
        rtol=1e-10,
        atol=1e-12,
    ):
        raise AssertionError(
            f"Friedman statistic mismatch for {scope} / {factor}."
        )

    if not np.isclose(
        row["p-value"],
        expected_p,
        rtol=1e-10,
        atol=1e-15,
    ):
        raise AssertionError(
            f"Friedman p-value mismatch for {scope} / {factor}."
        )

print("\nStatistical reproduction checks passed.")


## 11. Reproducibility notes

The formal dissertation experiment used:

- **Model:** `gpt-5.6-terra`
- **Reasoning mode:** standard
- **Reasoning effort:** low
- **Conversation state:** none; independent API calls
- **Storage flag:** `store=False`
- **Scenarios:** 10 formal FinTech scenarios
- **Artifacts:** User Story, Acceptance Criteria, Business Rules
- **Prompting approaches:** Zero-shot, Role-based, Structured, Few-shot
- **Generations per condition:** 3
- **Total generated outputs:** 360
- **Embedding model:** `text-embedding-3-large`
- **Stability measure:** mean pairwise cosine similarity across the three generations

The public workbook contains the experiment inputs, a blank API-results reproduction template, the original condition-level similarity results, aggregated evaluation results, and statistical analysis. Raw participant-level evaluator records and the original 360 generated texts are intentionally excluded from the public release.

To reproduce generation or embedding calculations, use your own OpenAI API key and explicitly enable the corresponding safety switches in this notebook.


The public notebook also reproduces the main Friedman, Wilcoxon, Holm-adjusted, Kendall's W, and rank-biserial analyses from the aggregated condition-level human-evaluation data. ICC is reported but not recalculated publicly because doing so requires evaluator-level records that are excluded from the public release.
